In [8]:
%%writefile src/preprocessing.py
# Your preprocessing code here


Overwriting src/preprocessing.py


In [9]:
%%writefile src/train_models.py
# Your training + evaluation code here


Overwriting src/train_models.py


In [4]:
%%writefile src/preprocessing.py
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

def preprocess_data(data):
    # Replace '?' with NaN and drop missing
    data = data.replace('?', pd.NA).dropna()

    # Encode categorical variables
    label_enc = LabelEncoder()
    for col in data.select_dtypes(include=['object']).columns:
        data[col] = label_enc.fit_transform(data[col])

    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(data.drop("income", axis=1))
    y = data["income"]
    return X, y

Writing src/preprocessing.py


In [5]:
!mkdir -p src

In [10]:
%%writefile src/train_models.py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

def train_and_evaluate(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(f"\n{name} Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(classification_report(y_test, y_pred))

        plt.savefig(f"results/{name}_confusion_matrix.png")

        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f"{name} - Confusion Matrix")
        plt.savefig(f"results/{name}_confusion_matrix.png")
        plt.close()

        plt.savefig("results/roc_curve.png")

        # ROC Curve
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:,1]
            fpr, tpr, _ = roc_curve(y_test, y_prob)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

    plt.plot([0,1],[0,1],'k--')
    plt.title("ROC Curve Comparison")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.savefig("results/roc_curve.png")
    plt.close()



Overwriting src/train_models.py
